In [1]:
%pip install gradio

  Using cached gradio-6.19.0-py3-none-any.whl.metadata (17 kB)
  Using cached anyio-4.14.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp314-cp314-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.138.2-py3-none-any.whl.metadata (26 kB)
  Using cached gradio_client-2.5.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
  Using cached orjson-3.11.9-cp314-cp314-win_amd64.whl.metadata (43 kB)
  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pydantic-2.13.4-py3-none-a


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

   ------------- -------------------------- 1/3 [jupyterlab_widgets]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\User\\Desktop\\ai_academy\\mslearn2\\document-inteligence\\ocr\\venv\\share\\jupyter\\labextensions\\@jupyter-widgets\\jupyterlab-manager\\static\\vendors-node_modules_d3-color_src_color_js-node_modules_d3-format_src_defaultLocale_js-node_m-09b215.2643c43f22ad111f4f82.js'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gradio as gr

def greet(name): 
    return "Hello " + name + "!"

demo = gr.Interface(fn = greet, inputs="text", outputs="text")
# demo = gr.Interface(fn = greet, inputs="audio", outputs="text")   # input을 audio로 입력
# demo = gr.Interface(fn = greet, inputs="text", outputs="audio")   # output을 audio로 출력
# demo.launch()
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://03f627ca41aeb19c95.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import gradio as gr
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv()

# .env에서 엔드포인트/키 읽어오기
document_endpoint = os.environ.get("YOUR_FORM_RECOGNIZER_ENDPOINT")
api_key = os.environ.get("YOUR_FORM_RECOGNIZER_KEY")
api_version = "api-version=2024-11-30"
# import os
# from dotenv import load_dotenv

# load_dotenv()
        
#         # 환경변수에서 값 읽어오기
# document_endpoint = os.environ.get("YOUR_FORM_RECOGNIZER_ENDPOINT")
# api_key = os.environ.get("YOUR_FORM_RECOGNIZER_KEY")
# api_version = "api-version=2024-11-30"

# endpoint = document_endpoint + api_version

# print("엔드포인트:", endpoint)
# print("키 로드 성공 여부:", bool(api_key))



def request_document_intelligence(image):
        # Azure Document Intelligence API 엔드포인트 및 인증 헤더 설정
        endpoint = document_endpoint.rstrip("/") + "/documentintelligence/documentModels/prebuilt-read:analyze?" + api_version
        
        headers = {
                   "Ocp-Apim-Subscription-Key": api_key,
                   "Content-Type":"image/png"
                  }
        
        with open(image, "rb") as f:
                image_data = f.read()
                
        response = requests.post(endpoint, headers=headers, data=image_data)
                
        if response.status_code != 202:
                print("요청중 에러남!")
                print(f"응답 코드: {response.status_code}")
                print(f"{response.text}")
                
                return None
        
        else: 
                print("OCR 잘함!")
                url = response.headers['Operation-Location']
                print(f"{url}")

        # 결과가 준비될 때까지 반복적으로 상태 확인
        while True:
                result_response = requests.get(url, headers=headers)
                        
                if result_response.status_code != 200:
                        print("Error: {result_response.status_code}")
                        return None
                        
                result_response_json = result_response.json()
                current_status = result_response_json.get("status")
                
                # 분석이 아직 진행 중이면 계속 대기
                if current_status == "running":
                        time.sleep(1) # 1초 대기 후 다시 확인
                        print("Current status: {}".format(current_status))
                        continue
                else: 
                        break
                
        # 분석이 성공적으로 끝난 경우 결과 반환, 아니면 None 반환
        if current_status == "Succeded":
                return result_response_json
        else: 
                return None
        
def converted_image(image_path, ocr_response_data):
        from PIL import Image, ImageDraw,ImageFont
        
        image = Image.open(image_path)
        draw = ImageDraw.Draw(image)
        
        block_list = ocr_response_data['analyzeResult']['paragraphs']['ImageFont']
        
        for block in block_list:
                line_color = (255,255,255)
                font = ImageFont.load_default(size=20)
                polygon = block['boundingRegions'][0]['polygon']
                content = block['content']
                polygon_point_list = [(polygon[i], polygon[i+1]) for i in range(0,len(polygon),2)]
                draw.polygon(polygon_point_list, outline=line_color, width=2)
                draw.text((polygon[0], polygon[1] - 20), content, fill=line_color, font=font)


def change_image(image_path):
        response_data=request_document_intelligence(image_path)
        converted_image = draw_image(image_path, response_data)
        print(f"chang_image 함수 OCR 결과 잘 받아봄.")
        return image

# Gradio UI 구성
with gr.Blocks() as demo:
        with gr.Row():
                input_image = gr.Image(label="이미지 선택", type="filepath", width=500)
                output_image = gr.Image(label="결과 이미지", type="pil", interactive=False, width=500)

                input_image.change(change_image, inputs=[input_image], outputs=[output_image])        
demo.launch()             

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


OCR 잘함!
https://sesac-022-document-inteligence.cognitiveservices.azure.com/documentintelligence/documentModels/prebuilt-read/analyzeResults/d517d6ed-38ac-4453-aabd-26dc6592538d?api-version=2024-11-30
Current status: running


Traceback (most recent call last):
  File "c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\blocks.py", line 1657, in call_function
   

In [ ]:
import os
import time
import mimetypes

import gradio as gr
import requests
from dotenv import load_dotenv
from PIL import Image, ImageDraw, ImageFont

load_dotenv()

# 환경변수에서 값 읽어오기
DOCUMENT_ENDPOINT = os.environ.get("YOUR_FORM_RECOGNIZER_ENDPOINT", "").rstrip("/")
API_KEY = os.environ.get("YOUR_FORM_RECOGNIZER_KEY", "")
API_VERSION = "2024-11-30"

print("엔드포인트:", DOCUMENT_ENDPOINT)
print("키 로드 성공 여부:", bool(API_KEY))


def request_document_intelligence(image_path):
    """Azure Document Intelligence(prebuilt-read)로 OCR 요청 후 결과를 반환"""
    if not DOCUMENT_ENDPOINT or not API_KEY:
        print("엔드포인트 또는 키가 설정되지 않았습니다. .env 파일을 확인하세요.")
        return None

    url = f"{DOCUMENT_ENDPOINT}/documentintelligence/documentModels/prebuilt-read:analyze?api-version={API_VERSION}"

    content_type = mimetypes.guess_type(image_path)[0] or "application/octet-stream"
    headers = {
        "Ocp-Apim-Subscription-Key": API_KEY,
        "Content-Type": content_type,
    }

    with open(image_path, "rb") as f:
        image_data = f.read()

    response = requests.post(url, headers=headers, data=image_data)

    if response.status_code != 202:
        print("요청중 에러남!")
        print(f"응답 코드: {response.status_code}")
        print(response.text)
        return None

    print("OCR 잘함!")
    operation_url = response.headers["Operation-Location"]
    print(operation_url)

    # 결과가 준비될 때까지 반복적으로 상태 확인
    while True:
        result_response = requests.get(operation_url, headers=headers)

        if result_response.status_code != 200:
            print(f"Error: {result_response.status_code}")
            return None

        result_response_json = result_response.json()
        current_status = result_response_json.get("status")

        # 분석이 아직 진행 중이면 계속 대기
        if current_status == "running" or current_status == "notStarted":
            print(f"Current status: {current_status}")
            time.sleep(1)  # 1초 대기 후 다시 확인
            continue
        else:
            break

    # 분석이 성공적으로 끝난 경우 결과 반환, 아니면 None 반환
    if current_status == "succeeded":
        return result_response_json
    else:
        print(f"분석 실패. 최종 상태: {current_status}")
        return None


def draw_ocr_result(image_path, ocr_response_data):
    """OCR 결과(문단 경계 상자 + 텍스트)를 이미지 위에 그려서 반환"""
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    paragraphs = ocr_response_data.get("analyzeResult", {}).get("paragraphs", [])
    font = ImageFont.load_default(size=20)

    for block in paragraphs:
        line_color = (255, 0, 0)
        polygon = block["boundingRegions"][0]["polygon"]
        content = block["content"]
        polygon_point_list = [(polygon[i], polygon[i + 1]) for i in range(0, len(polygon), 2)]
        draw.polygon(polygon_point_list, outline=line_color, width=2)
        draw.text((polygon[0], polygon[1] - 20), content, fill=line_color, font=font)

    return image


def change_image(image_path):
    if image_path is None:
        return None

    response_data = request_document_intelligence(image_path)
    if response_data is None:
        print("OCR 결과를 받지 못했습니다.")
        return Image.open(image_path)

    result_image = draw_ocr_result(image_path, response_data)
    print("change_image 함수 OCR 결과 잘 받아봄.")
    return result_image


# Gradio UI 구성
with gr.Blocks() as demo:
    with gr.Row():
        input_image = gr.Image(label="이미지 선택", type="filepath", width=500)
        output_image = gr.Image(label="결과 이미지", type="pil", interactive=False, width=500)

        input_image.change(change_image, inputs=[input_image], outputs=[output_image])

demo.launch()

엔드포인트: https://sesac-022-document-inteligence.cognitiveservices.azure.com
키 로드 성공 여부: True
* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.


c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


OCR 잘함!
https://sesac-022-document-inteligence.cognitiveservices.azure.com/documentintelligence/documentModels/prebuilt-read/analyzeResults/da19d67c-9985-4e98-8479-5687111f2254?api-version=2024-11-30
Current status: succeeded
change_image 함수 OCR 결과 잘 받아봄.


In [ ]:
import gradio as gr
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv()

# .env에서 엔드포인트/키 읽어오기
document_endpoint = os.environ.get("YOUR_FORM_RECOGNIZER_ENDPOINT")
api_key = os.environ.get("YOUR_FORM_RECOGNIZER_KEY")
api_version = "api-version=2024-11-30"
# import os
# from dotenv import load_dotenv

# load_dotenv()

#         # 환경변수에서 값 읽어오기
# document_endpoint = os.environ.get("YOUR_FORM_RECOGNIZER_ENDPOINT")
# api_key = os.environ.get("YOUR_FORM_RECOGNIZER_KEY")
# api_version = "api-version=2024-11-30"

# endpoint = document_endpoint + api_version

# print("엔드포인트:", endpoint)
# print("키 로드 성공 여부:", bool(api_key))



def request_document_intelligence(image):
        # Azure Document Intelligence API 엔드포인트 및 인증 헤더 설정
        # [주의] endpoint에 실제 도메인이 없고 key도 비어 있음 → 실제 호출 전에 본인 값으로 채워야 함 (기능 수정 범위 밖)
        endpoint = document_endpoint.rstrip("/") + "/documentintelligence/documentModels/prebuilt-read:analyze?" + api_version
        
        headers = {
                   "Ocp-Apim-Subscription-Key": api_key,
                   "Content-Type":"image/png"
                  }

        with open(image, "rb") as f:
                image_data = f.read()

        response = requests.post(endpoint, headers=headers, data=image_data)

        if response.status_code != 202:
                print("요청중 에러남!")
                print(f"응답 코드: {response.status_code}")
                print(f"{response.text}")

                return None

        else:
                print("OCR 잘함!")
                url = response.headers['Operation-Location']
                print(f"{url}")

        # 결과가 준비될 때까지 반복적으로 상태 확인
        while True:
                result_response = requests.get(url, headers=headers)

                if result_response.status_code != 200:
                        print(f"Error: {result_response.status_code}")
                        return None

                result_response_json = result_response.json()
                current_status = result_response_json.get("status")

                # 분석이 아직 진행 중이면 계속 대기
                if current_status == "running":
                        time.sleep(1) # 1초 대기 후 다시 확인
                        print("Current status: {}".format(current_status))
                        continue
                else:
                        break

        # 분석이 성공적으로 끝난 경우 결과 반환, 아니면 None 반환
        if current_status == "succeeded": 
                return result_response_json
        else:
                return None

def draw_image(image_path, ocr_response_data):
        from PIL import Image, ImageDraw,ImageFont

        image = Image.open(image_path)
        draw = ImageDraw.Draw(image)

        block_list = ocr_response_data['analyzeResult']['paragraphs'] 

        for block in block_list:
                line_color = (125,255,125)
                font = ImageFont.load_default(size=20)
                polygon = block['boundingRegions'][0]['polygon']
                content = block['content']
                polygon_point_list = [(polygon[i], polygon[i+1]) for i in range(0,len(polygon),2)]
                draw.polygon(polygon_point_list, outline=line_color, width=2)
                draw.text((polygon[0], polygon[1] - 20), content, fill=line_color, font=font)

        return image 


def change_image(image_path):
        response_data=request_document_intelligence(image_path)
        converted_image = draw_image(image_path, response_data) 
        print(f"chang_image 함수 OCR 결과 잘 받아봄.")
        return converted_image 

# Gradio UI 구성
with gr.Blocks() as demo:
        with gr.Row():
                input_image = gr.Image(label="이미지 선택", type="filepath", width=500)
                output_image = gr.Image(label="결과 이미지", type="pil", interactive=False, width=500)

                input_image.change(change_image, inputs=[input_image], outputs=[output_image])
demo.launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


OCR 잘함!
https://sesac-022-document-inteligence.cognitiveservices.azure.com/documentintelligence/documentModels/prebuilt-read/analyzeResults/032ca9a6-e3eb-4cad-bbd8-c8a6e9f9f748?api-version=2024-11-30
Current status: running
Current status: succeeded
chang_image 함수 OCR 결과 잘 받아봄.


In [ ]:
import gradio as gr
import requests
import time
import os
from dotenv import load_dotenv
 
load_dotenv()
 
# 환경변수에서 값 읽어오기
document_endpoint = os.environ.get("YOUR_FORM_RECOGNIZER_ENDPOINT")
api_key = os.environ.get("YOUR_FORM_RECOGNIZER_KEY")
api_version = "api-version=2024-11-30"
 
# [수정 10] 원본은 document_endpoint + api_version만 이어붙여서 "?"도 없고 실제 요청 경로(/documentintelligence/...)도 빠진 URL이 만들어지던 문제
#          → 실제 호출 경로까지 포함해서 완전한 URL로 조립
endpoint = document_endpoint.rstrip("/") + "/documentintelligence/documentModels/prebuilt-read:analyze?" + api_version
 
print("엔드포인트:", endpoint)
print("키 로드 성공 여부:", bool(api_key))


def request_document_intelligence(image):
        # Azure Document Intelligence API 엔드포인트 및 인증 헤더 설정

        headers = {
                   "Ocp-Apim-Subscription-Key": api_key,
                   "Content-Type":"image/png"
                  }
        
        with open(image, "rb") as f:
                image_data = f.read()

        response = requests.post(endpoint, headers=headers, data=image_data)

        if response.status_code != 202:
                print("요청중 에러남!")
                print(f"응답 코드: {response.status_code}")
                print(f"{response.text}")

                return None

        else:
                print("OCR 잘함!")
                url = response.headers['Operation-Location']
                print(f"{url}")

        # 결과가 준비될 때까지 반복적으로 상태 확인
        while True:
                result_response = requests.get(url, headers=headers)

                if result_response.status_code != 200:
                        print(f"Error: {result_response.status_code}")
                        return None

                result_response_json = result_response.json()
                current_status = result_response_json.get("status")

                # 분석이 아직 진행 중이면 계속 대기
                if current_status == "running":
                        time.sleep(1) # 1초 대기 후 다시 확인
                        print("Current status: {}".format(current_status))
                        continue
                else:
                        break

        # 분석이 성공적으로 끝난 경우 결과 반환, 아니면 None 반환
        if current_status == "succeeded": 
                return result_response_json
        else:
                return None

def draw_image(image_path, ocr_response_data):
        from PIL import Image, ImageDraw,ImageFont

        image = Image.open(image_path)
        draw = ImageDraw.Draw(image)

        block_list = ocr_response_data['analyzeResult']['paragraphs'] 

        for block in block_list:
                line_color = (125,255,125)
                font = ImageFont.load_default(size=20)
                polygon = block['boundingRegions'][0]['polygon']
                content = block['content']
                polygon_point_list = [(polygon[i], polygon[i+1]) for i in range(0,len(polygon),2)]
                draw.polygon(polygon_point_list, outline=line_color, width=2)
                draw.text((polygon[0], polygon[1] - 20), content, fill=line_color, font=font)

        return image 


def change_image(image_path):
        response_data=request_document_intelligence(image_path)
        converted_image = draw_image(image_path, response_data) 
        print(f"chang_image 함수 OCR 결과 잘 받아봄.")
        return converted_image 

# Gradio UI 구성
with gr.Blocks() as demo:
        with gr.Row():
                input_image = gr.Image(label="이미지 선택", type="filepath", width=500)
                output_image = gr.Image(label="결과 이미지", type="pil", interactive=False, width=500)

                input_image.change(change_image, inputs=[input_image], outputs=[output_image])
demo.launch()

엔드포인트: https://sesac-022-document-inteligence.cognitiveservices.azure.com/documentintelligence/documentModels/prebuilt-read:analyze?api-version=2024-11-30
키 로드 성공 여부: True
* Running on local URL:  http://127.0.0.1:7892
* To create a public link, set `share=True` in `launch()`.


c:\Users\User\Desktop\ai_academy\mslearn2\document-inteligence\ocr\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


OCR 잘함!
https://sesac-022-document-inteligence.cognitiveservices.azure.com/documentintelligence/documentModels/prebuilt-read/analyzeResults/c6dd1c65-0f62-4a78-b660-20113e8ae6d7?api-version=2024-11-30
Current status: succeeded
chang_image 함수 OCR 결과 잘 받아봄.
